# Parse all data into single dataframe

- Read all JSON files
- Read image files and extract central pixel values only

## Module Imports

In [2]:
import tarfile
import json
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import torch
import io
from matplotlib import pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import random

## Filepath

## Process JSON Files

In [26]:
def processTarJSON(path, jsonType):
    ''' takes the path of 1 tar files and JSON file type (auxiliary or text) and returns a dataframe of all json data'''
    rows = []
    with tarfile.open(path, "r") as tar:
        for member in tar:
            if member.isfile() and member.name.endswith(jsonType+'.json'):
                f = tar.extractfile(member)
                if f:
                    obj = json.loads(f.read().decode("utf-8"))
                    ID = member.name.split('\\')[-1].split('.')[0]
                    obj['ID'] = int(ID)
                    rows.append(obj)

    df = pd.DataFrame(rows)
    df['ID'] = pd.to_numeric(df['ID'], errors='coerce').astype(np.int32)
    
    return df

## Process Image Files

- Reads the central pixel only

In [4]:
def process_tarImg(path):
    ''' takes the path of 1 tar file and returns a dataframe of all image data for the center pixel'''
    data = {'ID':[], 'imgData':[]}
    with tarfile.open(path, "r") as tar:
        for member in tar:
            if member.isfile() and member.name.endswith(".pth"):
                # Extract file as bytes
                extracted = tar.extractfile(member)
                
                # Read the file into a BytesIO buffer
                buffer = io.BytesIO(extracted.read())

                imgData = torch.load(buffer, map_location="cpu")

                middlePixel = imgData[:, :, 2, 2]
                
                data['ID'].append(member.name.split('.')[0])
                data['imgData'].append(middlePixel)

    df = pd.DataFrame(data)
    df['ID'] = pd.to_numeric(df['ID'], errors='coerce').astype(np.int32)
    return df

## Process Tar Files
- Wrapper for previous two functions

In [22]:
def processTarJSONImg(path):
    '''Takes the path of 1 tar file and returns a dataframe with all data, ignoring missing values'''

    JSONTypes = ['.auxiliary', '.text']

    dfs = [] #list of 3 dataframes from each type of file in one tar file
    
    #allMissingData = []
    for JSONType in JSONTypes:
        dfJSON = processTarJSON(path, JSONType)
        dfs.append(dfJSON)
    
    dfs.append(process_tarImg(path))

    #merge 3 dataframes by ID
    dfTemp = dfs[0].merge(dfs[1], on='ID')
    df = dfTemp.merge(dfs[2], on='ID')

    id_col = df.pop('ID')
    df.insert(0, 'ID', id_col)

    #convert float64 and int64 to float32 to save memory
    df[df.select_dtypes(np.float64).columns] = df.select_dtypes(np.float64).astype(np.float32)
    df[df.select_dtypes(np.int64).columns] = df.select_dtypes(np.int64).astype(np.float32)

    df = df.drop(columns = ['sample_id'])

    return df

# Test on Single Tar
- With missing data

In [23]:
singleTarPath = 'GlobalGeoTree-6M\\GlobalGeoTree-6M\\GGT_4450001_4500000-000000.tar'

dfOneTar = processTarJSONImg(singleTarPath)

In [14]:
dfOneTar['imgData'].iloc[0].shape

torch.Size([12, 10])

In [18]:
dfOneTar.memory_usage(deep=True).sum() / 1024**2  # in MB

np.float64(33.69838809967041)

In [16]:
oneImg = dfOneTar['imgData'].iloc[0]

# Run on all Tars

In [3]:
tarFilePath = Path("GlobalGeoTree-6M/GlobalGeoTree-6M/")
chunksFilePath = Path("GlobalGeoTree-6M/Chunks1Pixel/")

In [ ]:


tarFiles = list(tarFilePath.glob("*.tar"))

for i, tarFile in tqdm(enumerate(tarFiles), total=len(tarFiles), desc = 'Processing Tar Files'):
    chunk_path = chunksFilePath / f'Chunk-{i}.pth'
    if chunk_path.exists():
        continue
     
    dfTemp = processTarJSONImg(tarFile)
    torch.save(dfTemp, chunk_path)

Processing Tar Files: 100%|██████████| 118/118 [37:55<00:00, 19.29s/it]


# Loading Data Back In

In [5]:
np.random.seed(0)

# Sort by the integer after 'Chunk-'
chunk_files = sorted(
    chunksFilePath.glob("Chunk-*.pth"),
    key=lambda p: int(p.stem.split("-")[1]) if "-" in p.stem else p.stem
)

sampled = []

for p in tqdm(chunk_files, desc="Sampling chunks"):
    df_chunk = torch.load(p, map_location="cpu", weights_only=False)
    if not isinstance(df_chunk, pd.DataFrame):
        continue
    frac = 0.05  # 5% from each chunk; adjust as needed
    if frac <= 0:
        continue
    take = df_chunk.sample(frac=frac)
    sampled.append(take)

df_sample = pd.concat(sampled, ignore_index=True)
print(df_sample.shape)

Sampling chunks: 100%|██████████| 118/118 [04:47<00:00,  2.44s/it]

(310981, 33)


In [6]:
torch.save(df_sample, chunksFilePath / 'SampledData-5Percent.pth')